# Local RAG Pipeline with HuggingFace Embeddings (Python 3.12)
This notebook demonstrates how to:
- Read local `.txt` and `.pdf` files
    - This is good enough for 'playing' but production would need  many more examples of files AND longer files.
    - A single service would not be enough

In [1]:
%pip install --upgrade pip

%pip install -q -r requirements.txt

Note: you may need to restart the kernel to use updated packages.


Note: you may need to restart the kernel to use updated packages.


In [2]:
from pathlib import Path
from config import settings

In [3]:
docs_path: Path = settings.DATA_DIR
processed_path: Path = settings.PROCESSED_FILE

print(settings)

Settings(
  BASE_PATH=.
  DATA_DIR=data_sources
  API_KEY=***
  PROCESSED_FILE=processed_data.txt
)


## Base Strategy Interface

In [4]:
from pathlib import Path
from abc import ABC, abstractmethod


class FileHandler(ABC):
    @abstractmethod
    def can_handle(self, file_path: Path) -> bool: ## How robust is this?
        pass

    @abstractmethod
    def extract_text(self, file_path: Path) -> str: ## How robust is this? How do we know the text is extracted correctly?
        pass

## The DocumentLoader

In [5]:
class DocumentLoader:
    def __init__(self):
        self.handlers: list[FileHandler] = []

    def register_handler(self, handler: FileHandler):
        self.handlers.append(handler)

    def extract_text(self, file_path: Path) -> str:
        for handler in self.handlers:
            if handler.can_handle(file_path): ## check if the handler can handle the file type
                ## if yes, call the extract_text method of the handler
                return handler.extract_text(file_path)
        raise ValueError(f"No handler for file type: {file_path.suffix}")

## Text Handler

In [6]:
class TxtHandler(FileHandler):
    def can_handle(self, file_path: Path) -> bool:
        return file_path.suffix.lower() == ".txt"

    def extract_text(self, file_path: Path) -> str:
        return file_path.read_text(encoding="utf-8")

## PDF Handler
Note that PDFs can be encoded with text or as 'images' of text. This class checks and handles both cases

In [7]:
import PyPDF2
from pdf2image import convert_from_path
import pytesseract


class PdfHandler(FileHandler):
    def can_handle(self, file_path: Path) -> bool:
        return file_path.suffix.lower() == ".pdf"

    def is_text_based(self, file_path: Path) -> bool:
        try:
            with open(file_path, "rb") as f:
                reader = PyPDF2.PdfReader(f)
                return any(page.extract_text() for page in reader.pages)
        except:
            return False

    def extract_text(self, file_path: Path) -> str:
        if self.is_text_based(file_path):
            with open(file_path, "rb") as f:
                reader = PyPDF2.PdfReader(f)
                return "\n".join(page.extract_text() or "" for page in reader.pages)
        else:
            pages = convert_from_path(file_path)
            return "\n".join(pytesseract.image_to_string(p) for p in pages)

In [8]:
loader = DocumentLoader()
loader.register_handler(TxtHandler())
loader.register_handler(PdfHandler())

In [9]:
from dataclasses import dataclass
from typing import List, Dict

@dataclass
class TextFile:
    filename: str
    content: str


all_texts: List[TextFile] = []

In [10]:
for file in docs_path.glob("*"): ## Is this recursive?
    try:
        content = loader.extract_text(file)
        all_texts.append(TextFile(filename=file.name, content=content))
    except Exception as e:
        print(f"Error with {file.name}: {e}")

## Check the documents have been processed

In [11]:
print(f"Extracted {len(all_texts)} documents.")

Extracted 5 documents.


In [12]:
print(all_texts[1])

TextFile(filename='LHH Values Audit Exercise.pdf', content='Values\nAudit\nValues are powerful principles or qualifications \nthat underpin our actions and our opinions \nabout events and people. Sometimes they are \ncalled ‘critical needs’ . \nValues change significantly over the years and \nit’s important to be clear about them and to \nbetter assess how they can be met. Decisions \nwe make often reflect what’s important in our \nlives. If there’s a conflict between a decision \nand your own values, very often this conflict \ncan contribute to both personal and career \ndissatisfaction and unhappiness. \nSources of strong dissatisfaction often occur \nwhen we have cherished values that are not \ncurrently being acted on or when we have two \nstrong values that are in conflict, such as a \ndesire for achievement and leisure. Often we’re \nnot fully aware of values that drive us, especially \nif they have always been satisfied. \n2\n© LHH. All rights reserved\nValues Audit Exercise\nDi

In [13]:
with open(processed_path, "w", encoding="utf-8") as f: ## Open the file in write mode
    for doc in all_texts: ## Loop through the documents
        f.write(f"__META__FILE__NAME: {doc.filename}\n")
        f.write(f"__CONTENT__: {doc.content}\n")
        f.write("\n" + "="*80 + "\n\n")